# H02 — 亲手建立可持续成长的工程

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H02 — 亲手建立可持续成长的工程。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H02-project.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

今天从空目录建立自己的工程。以后加 Timer、传感器和网络，都在这个工程里进行，而不每课重建。

前置：H00–H01；空目录、Git。

## 本次够用的知识

根 CMakeLists.txt 加载 ESP-IDF；组件 CMakeLists.txt 登记源码与依赖；app_main 是 C 链接入口，C++ 文件用 extern "C" 保留名称。头文件声明不会自动把 .cpp 加入编译，编译成功也可能在链接阶段找不到实现。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 检查起点与工程地图 · Guided

读 ../device-template/README.md 和 H00 环境记录。选一个自己的目录名，确认目标目录为空，检查 VS Code 打开的是新工程而不是课程根的 host CMake。
先口述最小工程需要哪些文件；AGENTS.md 和 .academy/ 都不参与固件运行。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 从最小骨架开始 · Guided

首选亲手创建根 CMakeLists.txt、main/CMakeLists.txt 和 main/main.cpp，参照官方 build-system 与 ../device-template 的最小骨架。
也可运行 python3 tools/academy_tutor.py init-project ../my-desk-tool，在创建后逐行解释三份文件；复制不会计作独立创建验收。将模板 AGENTS.md 与 .academy 带入新工程，链接课程根，之后 H03–H14 继续使用这里。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 让源文件参与构建 · Guided

新增 main/device_name.hpp 和 main/device_name.cpp，将下面示例接入 main/main.cpp。main/CMakeLists.txt 的 SRCS 必须包含 "main.cpp" "device_name.cpp"；REQUIRES 包含 freertos log。
预测 header 只声明、不链接 source 会发生什么。build/flash/monitor 验证设备名，不预先替你修下一步的漏登记错误。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H02/examples/device_name.hpp`。目标：`main/device_name.hpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。

In [ ]:
%%esp32_example --target main/device_name.hpp
const char* device_name();


示例来源：`step2/lessons/H02/examples/device_name.cpp`。目标：`main/device_name.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/device_name.cpp
#include "device_name.hpp"

const char* device_name() {
    return "desk-tool";
}


示例来源：`step2/lessons/H02/examples/main.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"
#include "device_name.hpp"

extern "C" void app_main() {
    ESP_LOGI("desk_tool", "device=%s", device_name());
    while (true) {
        vTaskDelay(pdMS_TO_TICKS(1000));
    }
}


## 把模块换成自己的 · Modify

改设备名，再新增一个只返回固件版本的函数，声明和实现分开。自己登记源码并构建；检查 build/compile_commands.json 是否真的包含新 .cpp。
解释 sdkconfig.defaults、sdkconfig 与 build 的关系；menuconfig 更改本机配置，公共默认值要有意整理，不能将 Wi-Fi 凭据提交。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 制造一次链接错误 · Debug

暂时从 SRCS 删除 device_name.cpp，只保留调用与声明，运行 build。记录 undefined reference 一类诊断：它与缺少头文件或语法错误有什么不同？
恢复登记，再把函数名一边改错；通过诊断判断是声明、实现还是调用不一致。保存首个错误，不把整屏日志当成多个独立故障。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 把按钮行为带进来 · Build

仅根据 H01 接线和 STEP 1 自己的实现，在新工程加入 Board/Controller/Esp32Board；入口负责初始化与调用，不能全部塞进 main.cpp。
验收：关闭 Notebook，用官方命令独立 set-target、build、flash、monitor；解释每个源码文件如何被编译。host 逻辑测试仍可运行，不依赖课程 magic。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 保存第一份可恢复起点 · Open

git init，检查 .gitignore，再 git add 指定源码/配置/README/AGENTS.md；检查 diff 后提交。不要提交 sdkconfig、build、课程绝对路径或密码。dependencies.lock 若出现应按依赖复现需求提交，而不是当作缓存。
写一张工程地图并记录提交号。后续课程仅作增量修改；最后独立验收会在干净目录复现这个工程。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] 能够解释入口、两层 CMake、声明/实现/链接
- [ ] 自己添加 .cpp 并修复漏登记错误
- [ ] 官方命令能构建烧录，自己的按钮行为在新工程运行

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H02',
    [
        '能够解释入口、两层 CMake、声明/实现/链接',
        '自己添加 .cpp 并修复漏登记错误',
        '官方命令能构建烧录，自己的按钮行为在新工程运行',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H02/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-guides/build-system.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-guides/build-system.html)